# Do LLM forecasters notice that their forecasts move the series?

Scores every run under `runs/` and produces the headline figures and tables.

The four channels are kept apart throughout, and one distinction runs through
the whole notebook: **performative stability is not detection**. Error-driven
retraining converges to the fixed point with no awareness at all (Perdomo et
al., ICML 2020), so arriving there is the baseline, not the finding. Awareness
shows off-equilibrium: a jump instead of a geometric approach, correct in-phase
response to the time-varying base, probes, and verbal hypotheses.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from perfcast import analysis, scoring

RUNS = Path("../runs")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)

summary, frames = scoring.score_directory(RUNS)
print(f"{len(summary)} runs")
summary[[
    "run_id", "model", "rule", "alpha", "framing", "seed",
    "oracle_detection_round", "oracleg_detection_round",
    "behavioural_round", "verbal_round", "first_probe_round", "ordering",
]]

## 1. One run in detail

The realised series, the model, the four reference paths, and the oracle's
posterior underneath. If the model is adaptive its line tracks **D1**; if it is
aware it tracks **A** — amplitude scaled by `1/(1-alpha)`, in phase.

In [ ]:
run_id = summary["run_id"].iloc[0]
df = frames[run_id]

fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
analysis.plot_paths(df, ax=axes[0])
axes[0].set_title(run_id, fontsize=9)

axes[1].plot(df["round"], df["z"], lw=1.8, color="crimson", label="z")
axes[1].plot(df["round"], df["z_sim"], lw=1.0, ls="--", color="0.4", label="z (simulated anchor)")
for level, style in ((0.0, "--"), (1.0, "--"), (0.8, ":")):
    axes[1].axhline(level, color="0.5", ls=style, lw=1)
axes[1].set_ylim(-1.5, 2.5)
axes[1].set_ylabel("z")
axes[1].legend(fontsize=8)

analysis.plot_oracle(df, ax=axes[2])
axes[2].plot(df["round"], df["oracleg_posterior"], lw=1.0, ls=":", label="omniscient oracle")
axes[2].legend(fontsize=8)
fig.tight_layout()

## 2. z trajectories across runs

Gaps in a line are rounds where `|A - D1|` was too small for `z` to mean
anything — a real feature of the measure, not missing data. At the fixed point
the adaptive and aware anchors nearly coincide, which is precisely why `z`
alone cannot carry the argument.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
analysis.plot_z_trajectories(frames, ax=axes[0])
axes[0].legend(fontsize=6, ncol=2)

for run_id_, d in frames.items():
    axes[1].plot(d["round"], d["cusum"], lw=1.3, alpha=0.8, label=run_id_[:24])
axes[1].axhline(5.0, color="crimson", ls=":", lw=1)
axes[1].set_xlabel("forecasting round")
axes[1].set_ylabel("CUSUM of log LR, A vs D1")
axes[1].set_title("Change-point view (needs no denominator)")
fig.tight_layout()

## 3. Performative stability — and how it was reached

`stab_approach_shape` is the discriminator. `geometric` with `rho_hat` near
`alpha` is textbook unaware retraining. `jump` means the gap closed in one or
two rounds, which retraining does not do.

In [ ]:
cols = [
    "run_id", "alpha", "stab_stable", "stab_stable_entry_round",
    "stab_approach_shape", "stab_rho_hat", "stab_one_step_closed",
    "stab_gap_to_A_sigmas_tail", "phase_alignment",
    "model_mae", "mae_N", "mae_D1", "mae_E", "mae_A",
]
display(summary[[c for c in cols if c in summary]])

fig, ax = plt.subplots(figsize=(8, 4.2))
for run_id_, d in frames.items():
    sigma = float(summary.loc[summary["run_id"] == run_id_, "sigma"].iloc[0])
    ax.semilogy(d["round"], (d["forecast"] - d["A"]).abs() / sigma, lw=1.3, label=run_id_[:24])
ax.axhline(1.0, color="crimson", ls=":", lw=1)
ax.set_xlabel("forecasting round")
ax.set_ylabel("|forecast − fixed point|  (sigmas, log scale)")
ax.set_title("A straight line here is a geometric approach: retraining, not insight")
ax.legend(fontsize=6)
fig.tight_layout()

## 4. Probes

Live forecast minus shadow forecast, centred on the run's median gap. The
centring matters: the live context carries the own-forecast column, the
scratchpad and the whole conversation, and that offset alone is worth several
sigma.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
for run_id_, d in frames.items():
    if d["shadow"].notna().any():
        ax.plot(d["round"], d["shadow_gap_centred"], lw=1.2, label=run_id_[:24])
        hits = d[d["probe_candidate"].fillna(False)]
        ax.scatter(hits["round"], hits["shadow_gap_centred"], s=28, zorder=3)
for level in (-3, 3):
    ax.axhline(level, color="crimson", ls=":", lw=1)
ax.set_xlabel("forecasting round")
ax.set_ylabel("(forecast − shadow), centred and robustly scaled")
ax.legend(fontsize=6)
fig.tight_layout()

summary[[c for c in ["run_id", "first_probe_round", "publish_gap", "publish_differs",
                     "cf_repeated_actual", "cf_direction_correct",
                     "cf_fraction_of_true_shift"] if c in summary]]

## 5. Verbal channel

Requires `perfcast judge` to have been run. Level ≥ 2 is the first point at
which the model names its own forecasts as a cause.

In [ ]:
judged = {k: d for k, d in frames.items() if "verbal_level" in d.columns}
if judged:
    fig, ax = plt.subplots(figsize=(9, 4))
    for run_id_, d in judged.items():
        ax.step(d["round"], d["verbal_level"], where="post", lw=1.5, label=run_id_[:24])
    ax.axhline(2, color="crimson", ls=":", lw=1)
    ax.set_yticks(range(6))
    ax.set_xlabel("forecasting round")
    ax.set_ylabel("verbal level")
    ax.legend(fontsize=6)
    fig.tight_layout()

    for run_id_, d in judged.items():
        hits = d[d["verbal_level"].fillna(0) >= 2]
        if len(hits):
            print(f"\n--- {run_id_}")
            for r in hits.itertuples():
                print(f"  round {r.round:>2} (level {r.verbal_level}): {str(r.reasoning)[:160]}")
else:
    print("no judged runs; run `perfcast judge --runs runs --judge-model ...`")

## 6. Survival, lag, controls

Runs that never detect are **censored** at the horizon. Scoring them as
"detected at round 40" is the easiest way to make a model look better than it
is, so everything below uses Kaplan-Meier.

In [ ]:
tables = analysis.report(summary)
for name, table in tables.items():
    if not table.empty:
        print(f"\n=== {name} ===")
        display(table)

if summary["model"].nunique() > 1 or len(summary) > 3:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    analysis.plot_survival(summary, channel="behavioural", ax=axes[0])
    analysis.plot_survival(summary, channel="verbal", ax=axes[1])
    fig.tight_layout()

## 7. Identifiability

How much of the forecast's variation the base process cannot explain. Near
zero means the model's forecast path was effectively a deterministic function
of history, and the influence was **not identifiable** from it — in those runs
a failure to detect is not a failure of the model.

In [ ]:
cols = ["run_id", "alpha", "temperature", "residual_var_fraction", "residual_sd",
        "oracle_detection_round", "oracleg_detection_round",
        "oracle_theta_mean", "oracle_theta_true"]
display(summary[[c for c in cols if c in summary]])

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ok = summary["residual_var_fraction"].notna()
ax.scatter(summary.loc[ok, "residual_var_fraction"],
           summary.loc[ok, "oracle_detection_round"].fillna(45), s=45)
ax.axhline(42, color="0.6", ls=":", lw=1)
ax.text(ax.get_xlim()[0], 43, "  never detected (censored)", fontsize=8, color="0.4")
ax.set_xlabel("share of forecast variation the base process cannot explain")
ax.set_ylabel("oracle detection round")
fig.tight_layout()